# Phase 6 · 04 — Adapters: Houlsby vs Pfeiffer

Two experiments in one cell, and the task statement lists them as two separate
techniques. **Houlsby** puts an adapter after *every* sublayer, so it can correct
the attention output before the FFN sees it. **Pfeiffer** puts *one* after the FFN,
so it can only correct what leaves the block — roughly half the parameters.

Both zero-initialise the up-projection, so the adapted model **is** the frozen
model at step 0, asserted on real logits before training.

## 1 — Setup

In [8]:
import os, sys, json, math, time, random, pickle, re, zipfile, warnings
from pathlib import Path
from dataclasses import dataclass, asdict, field, replace
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
warnings.filterwarnings("ignore")

# ───────────────────────── EDIT THESE ─────────────────────────
# HF_TOKEN comes from your Colab secret: key icon in the left sidebar ->
# add HF_TOKEN -> enable "Notebook access". Nothing is written into this file,
# so the notebook is safe to keep in Drive or hand in.
CKPT_NAME  = "best_encdec.pt"   # your Phase 1 encoder-decoder checkpoint on Drive
OUT_OVERRIDE = None    # None -> /content/drive/MyDrive/cs781_phase6
BUDGET_MIN = 90        # wall-clock cap PER experiment
QUICK      = False     # True = tiny smoke run of the whole notebook
# ──────────────────────────────────────────────────────────────

# Colab + Google Drive only. No environment detection: everything lives in
# Drive, so a run survives a disconnect and `resume` finds it next time.
from google.colab import drive as _gdrive
_gdrive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive")
if not DRIVE.exists():
    raise RuntimeError("Drive did not mount — re-run this cell and approve the "
                       "prompt. Everything this notebook writes goes to Drive, "
                       "so nothing should run until it is mounted.")

OUT = Path(OUT_OVERRIDE) if OUT_OVERRIDE else DRIVE / "cs781_phase6"
SEARCH = [DRIVE, Path("/content"), Path(".")]
for sub in ("cache", "models", "analysis", "plots", "submissions", "experiments"):
    (OUT / sub).mkdir(parents=True, exist_ok=True)


def _hf_token():
    """From the Colab secret HF_TOKEN, falling back to $HF_TOKEN."""
    try:
        from google.colab import userdata
        t = userdata.get("HF_TOKEN")
        if t:
            print("HF token    : from the Colab secret HF_TOKEN")
            return t
    except Exception as e:
        print(f"HF token    : Colab secret unavailable ({type(e).__name__})")
    t = os.environ.get("HF_TOKEN", "")
    print("HF token    : " + ("from $HF_TOKEN" if t else
          "NOT FOUND — add a Colab secret named HF_TOKEN with 'Notebook "
          "access' enabled, or the dataset download will fail"))
    return t


def find_file(name, roots=None, required=True):
    for root in (roots or SEARCH):
        root = Path(root)
        if not root.exists():
            continue
        direct = root / name
        if direct.exists():
            return direct
        for p in root.rglob(name):
            return p
    if required:
        raise FileNotFoundError(
            f"could not find {name!r} under {[str(r) for r in (roots or SEARCH)]}")
    return None


TOK_PY   = find_file("a1_tokenizer.py")
TOK_JSON = find_file("bpe_40k.json")

# The checkpoint is whatever you named it. CKPT_NAME is tried first; the rest are
# fallbacks so this works whether the file is best_encdec.pt, best_encdec (no
# extension), or the best.pt/final.pt the Phase 1 notebook wrote.
CKPT_P1 = None
for _cand in (CKPT_NAME, "best_encdec.pt", "best_encdec", "best_encdec.pth",
              "encdec_best.pt", "best.pt", "final.pt"):
    CKPT_P1 = find_file(_cand, required=False)
    if CKPT_P1 is not None:
        break
if CKPT_P1 is None:
    seen = []
    for _r in SEARCH:
        if Path(_r).exists():
            seen += [str(p) for p in Path(_r).rglob("*.pt")][:20]
    raise FileNotFoundError(
        f"no Phase 1 encoder-decoder checkpoint found. Set CKPT_NAME at the top "
        f"of this cell to one of the .pt files that are actually present:\n  "
        + "\n  ".join(seen[:20] or ["(none found at all — is Drive mounted?)"]))

sys.path.insert(0, str(TOK_PY.parent))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"outputs   : {OUT}")
print("            (Drive — survives disconnects, and `resume` reads "
      "checkpoints back from here)")
print(f"tokenizer : {TOK_PY}  |  {TOK_JSON}")
print(f"phase 1   : {CKPT_P1}  ({CKPT_P1.stat().st_size/1e6:.0f} MB)")
HF_TOKEN = _hf_token()
print(f"torch     : {torch.__version__}  device {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU       : {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB)")
else:
    print("WARNING: no GPU — this phase is not feasible on CPU")
print(f"\nper-experiment wall-clock cap: {BUDGET_MIN} min")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
outputs   : /content/drive/MyDrive/cs781_phase6
            (Drive — survives disconnects, and `resume` reads checkpoints back from here)
tokenizer : /content/drive/MyDrive/a1_tokenizer.py  |  /content/drive/MyDrive/bpe_40k.json
phase 1   : /content/drive/MyDrive/best_encdec.pt  (927 MB)
HF token    : from the Colab secret HF_TOKEN
torch     : 2.11.0+cu130  device cuda
GPU       : NVIDIA L4 (24 GB)

per-experiment wall-clock cap: 90 min


## 2 — Library

The next six cells are **identical in every Phase 6 notebook**, so whichever one
you open is self-contained. Run them once per session; you can collapse them.

They need two things from your Drive: `a1_tokenizer.py` with `bpe_40k.json`, and
your Phase 1 encoder-decoder checkpoint (`CKPT_NAME` at the top of the setup cell,
default `best_encdec.pt`).

In [9]:
@dataclass
class P6Config:
    # paths
    out_dir: str = "."
    cache_dir: str = "./cache"
    results_json: str = "./results.json"
    ckpt_p1: str = "./best.pt"
    tokenizer_json: str = "./bpe_40k.json"

    # Phase 1 architecture — read from the checkpoint, not guessed
    d_model: int = 640
    enc_layers: int = 5
    dec_layers: int = 5
    n_heads: int = 10
    n_kv_heads: int = 5
    d_ff: int = 1728
    rope_theta: float = 10000.0
    rms_eps: float = 1e-5
    dropout: float = 0.0
    vocab_size: int = 40000

    # Sequence budget. Measured on the real Phase 6 corpus (7,030 pairs):
    # documents median ~4,260 tokens, summaries median ~870, p90 ~2,050.
    # tgt_len=256 would truncate 97% of the references and teach the model to
    # stop far too early, so the default is 1024, which covers ~61% whole.
    # Phase 1 pretrained at 512/512, so positions 512-1023 are extrapolated by
    # RoPE rather than learned — that is the trade this default makes, and it is
    # worth a line in the report.
    src_len: int = 1024          # document tokens into the encoder
    tgt_len: int = 1024          # summary tokens out of the decoder
    doc_strategy: str = "head"   # head | head_tail | lead_k

    # training
    lr: float = 3e-5
    head_lr: float = 3e-5
    weight_decay: float = 0.01
    epochs: int = 12
    batch_size: int = 2
    grad_accum: int = 8
    warmup_frac: float = 0.05
    max_grad_norm: float = 1.0
    resume: bool = True          # continue a previous run of the same name
    label_smoothing: float = 0.1
    seed: int = 42
    eval_every: int = 250
    patience: int = 6
    max_steps: int | None = None
    max_minutes: float | None = None    # wall-clock cap; the real guard

    # generation
    gen_max_new: int = 1024
    gen_min_new: int = 96
    gen_beams: int = 1              # 1 = greedy
    gen_len_penalty: float = 1.0
    gen_no_repeat_ngram: int = 3

    # Phase 6 ships train + test only. When no dev file exists, a held-out
    # slice of train becomes dev — deterministic, seeded, and disjoint from the
    # training rows, so model selection never sees what it trained on.
    dev_frac: float = 0.05
    dev_max: int = 800

    # data size
    quick: bool = False
    max_train: int | None = None
    max_dev: int | None = None

    hf_repo: str = "Exploration-Lab/CS-781-Capstone"
    hf_token: str | None = None

    def __post_init__(self):
        self.head_dim = self.d_model // self.n_heads
        if self.quick:
            self.src_len = min(self.src_len, 256)
            self.tgt_len = min(self.tgt_len, 128)
            self.gen_max_new = min(self.gen_max_new, 128)
            self.max_train = self.max_train or 200
            self.max_dev = self.max_dev or 40
            self.epochs = 1
            self.eval_every = 25
            self.max_steps = self.max_steps or 60


def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)


PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3


def load_tokenizer(cfg):
    from a1_tokenizer import BPETokenizer
    tok = BPETokenizer.load(str(cfg.tokenizer_json))
    specials = getattr(tok, "specials", None) or getattr(tok, "special_tokens", None)
    print(f"tokenizer: vocab {tok.vocab_size:,}  specials {specials}")
    print(f"  PAD={PAD_ID} UNK={UNK_ID} BOS={BOS_ID} EOS={EOS_ID} "
          f"(</s> was never used in pretraining — this phase trains it)")
    return tok


def enc_ids(tok, text):
    """Phase 1 called tokenizer.encode(text) with no flag; be tolerant of either
    signature so this notebook works with the exact module on your drive."""
    try:
        return tok.encode(text, add_special_tokens=False)
    except TypeError:
        return tok.encode(text)


def dec_ids(tok, ids):
    ids = [int(i) for i in ids if int(i) not in (PAD_ID, BOS_ID, EOS_ID)]
    return tok.decode(ids)

### 2.2 Data — document/summary pairs, with the long-document strategy

In [10]:
def _hf(cfg, rel):
    from huggingface_hub import hf_hub_download
    return hf_hub_download(cfg.hf_repo, rel, repo_type="dataset",
                           token=cfg.hf_token)


P6_CANDIDATES = [
    "Phase_6_Summ/{sub}/{split}.json",          # the actual layout
    "Phase_6_Summarization/{sub}/{split}.json",
    "Phase_6_Abstractive_Summarization/{sub}/{split}.json",
    "Phase_6_Summarisation/{sub}/{split}.json",
    "Phase_6/{sub}/{split}.json",
]


def _load_raw(cfg, split, local=None):
    """Returns [(id, document, summary|None)]. The repo layout for Phase 6 is not
    pinned in the task statement, so several plausible paths are tried and the one
    that resolves is reported."""
    if local and os.path.exists(local):
        path = local
    else:
        sub = "Test" if split == "test" else "Train"
        path, errs = None, []
        for pat in P6_CANDIDATES:
            rel = pat.format(sub=sub, split=split)
            try:
                path = _hf(cfg, rel)
                print(f"  [{split}] {rel}")
                break
            except Exception as e:
                errs.append(f"{rel}: {type(e).__name__}")
        if path is None:
            raise FileNotFoundError(
                f"could not locate the Phase 6 {split} file. Tried:\n  "
                + "\n  ".join(errs)
                + "\nPass local=... with the downloaded path, or add the correct "
                  "pattern to P6_CANDIDATES.")
    with open(path) as f:
        raw = json.load(f)
    rows = raw.values() if isinstance(raw, dict) else raw
    out = []
    for r in rows:
        if not isinstance(r, dict):
            continue
        doc = r.get("document") or r.get("text") or r.get("input") or ""
        if isinstance(doc, list):
            doc = " ".join(map(str, doc))
        if not doc:
            continue
        summ = r.get("summary", r.get("target"))
        if isinstance(summ, list):
            summ = " ".join(map(str, summ))
        out.append((str(r.get("id", len(out))), doc, summ))
    return out


def select_source(text, tok, cfg):
    """Which part of a long judgment reaches the encoder.

    head      the opening, which in Indian judgments carries the facts and the
              question before the court;
    head_tail the opening plus the closing, since the holding sits at the end;
    lead_k    the first sentences only, the classic extractive prior.
    """
    L = cfg.src_len
    words = text.split()
    if cfg.doc_strategy == "lead_k":
        sents = re.split(r"(?<=[.!?])\s+", text)
        acc, n = [], 0
        for s in sents:
            k = len(s.split())
            if n + k > 2 * L:
                break
            acc.append(s); n += k
        ids = enc_ids(tok, " ".join(acc))[:L]
    elif cfg.doc_strategy == "head_tail":
        nh = L // 2
        h = enc_ids(tok, " ".join(words[: 2 * L]))
        if len(h) <= L:
            ids = h
        else:
            t = enc_ids(tok, " ".join(words[-2 * L:]))
            ids = h[:nh] + t[-(L - nh):]
    else:
        ids = enc_ids(tok, " ".join(words[: 2 * L]))[:L]
    return ids[:L]


def encode_pairs(tok, rows, cfg, desc=""):
    """src [N, src_len] padded; tgt_in / tgt_out [N, tgt_len] padded.
    tgt_in  = [BOS, y0 ... y_{n-1}]      tgt_out = [y0 ... y_{n-1}, EOS]
    Padded positions in tgt_out are PAD and are ignored by the loss."""
    N = len(rows)
    S = np.full((N, cfg.src_len), PAD_ID, dtype=np.int32)
    TI = np.full((N, cfg.tgt_len), PAD_ID, dtype=np.int32)
    TO = np.full((N, cfg.tgt_len), PAD_ID, dtype=np.int32)
    slen = np.zeros(N, dtype=np.int32); tlen = np.zeros(N, dtype=np.int32)
    t0 = time.time()
    for i, (_, doc, summ) in enumerate(rows):
        s = select_source(doc, tok, cfg)
        S[i, :len(s)] = s; slen[i] = max(len(s), 1)
        if summ is not None:
            y = enc_ids(tok, summ)[: cfg.tgt_len - 1]
            TI[i, 0] = BOS_ID
            TI[i, 1:len(y) + 1] = y
            TO[i, :len(y)] = y
            TO[i, len(y)] = EOS_ID          # the stop token this phase teaches
            tlen[i] = len(y) + 1
        if desc and i and i % 500 == 0:
            print(f"  {desc} {i}/{N} ({time.time()-t0:.0f}s)")
    if desc:
        print(f"  {desc} {N} pairs in {time.time()-t0:.0f}s")
    return S, TI, TO, slen, tlen


def prepare_data(cfg, tok, splits=("train", "dev"), local=None):
    local = local or {}
    Path(cfg.cache_dir).mkdir(parents=True, exist_ok=True)
    key = f"{cfg.doc_strategy}_{cfg.src_len}_{cfg.tgt_len}"
    out = {}
    for sp in splits:
        cap = cfg.max_train if sp == "train" else cfg.max_dev
        tag = f"{sp}_{key}" + (f"_n{cap}" if cap else "")
        f = Path(cfg.cache_dir) / f"p6_{tag}.npz"
        fm = Path(cfg.cache_dir) / f"p6_{tag}.json"
        if f.exists() and fm.exists():
            z = np.load(f); meta = json.loads(fm.read_text())
            out[sp] = (z["S"], z["TI"], z["TO"], z["slen"], z["tlen"],
                       meta["ids"], meta["summaries"])
            print(f"[cache hit ] {sp}: {len(meta['ids'])} pairs ({f.name})")
            continue
        rows = load_split(cfg, sp, local.get(sp))
        if cap and len(rows) > cap:
            rng = np.random.default_rng(cfg.seed)
            rows = [rows[i] for i in
                    sorted(rng.choice(len(rows), cap, replace=False))]
        print(f"[tokenizing] {sp}: {len(rows)} pairs, "
              f"{cfg.doc_strategy} src={cfg.src_len} tgt={cfg.tgt_len}")
        S, TI, TO, sl, tl = encode_pairs(tok, rows, cfg, desc=sp)
        np.savez_compressed(f, S=S, TI=TI, TO=TO, slen=sl, tlen=tl)
        fm.write_text(json.dumps({"ids": [r[0] for r in rows],
                                  "summaries": [r[2] for r in rows]}))
        out[sp] = (S, TI, TO, sl, tl, [r[0] for r in rows], [r[2] for r in rows])
    return out


class SumDataset(torch.utils.data.Dataset):
    def __init__(self, S, TI, TO):
        self.S, self.TI, self.TO = S, TI, TO
    def __len__(self):
        return len(self.S)
    def __getitem__(self, i):
        return (torch.from_numpy(self.S[i].astype(np.int64)),
                torch.from_numpy(self.TI[i].astype(np.int64)),
                torch.from_numpy(self.TO[i].astype(np.int64)), i)


def make_loader(S, TI, TO, bs, shuffle, seed=42, workers=2):
    g = torch.Generator(); g.manual_seed(seed)
    return torch.utils.data.DataLoader(
        SumDataset(S, TI, TO), batch_size=bs, shuffle=shuffle,
        num_workers=workers, pin_memory=(DEVICE.type == "cuda"), generator=g)


_SPLITS = {}


def load_split(cfg, split, local=None):
    """train / dev / test.

    Phase 6 provides train and test only — there is no dev.json in the dataset.
    Model selection still needs held-out data, so when no dev file resolves, a
    seeded slice of train is held out as dev and the remaining rows become train.
    The two are disjoint by construction, so nothing selected on dev was trained
    on, and the split is identical in every notebook because it depends only on
    cfg.seed."""
    if split == "test" or not cfg.dev_frac:
        return _load_raw(cfg, split, local)

    key = (cfg.seed, cfg.dev_frac, cfg.dev_max)
    if key not in _SPLITS:
        train = _load_raw(cfg, "train", (local or {}).get("train")
                          if isinstance(local, dict) else None)
        dev = None
        try:                                   # use a real dev file if there is one
            dev = _load_raw(cfg, "dev")
            print(f"  using the dataset's own dev split ({len(dev)} pairs)")
            _SPLITS[key] = (train, dev)
        except Exception:
            n_dev = min(max(int(len(train) * cfg.dev_frac), 20), cfg.dev_max)
            perm = np.random.default_rng(cfg.seed).permutation(len(train))
            di, ti = sorted(perm[:n_dev]), sorted(perm[n_dev:])
            _SPLITS[key] = ([train[i] for i in ti], [train[i] for i in di])
            print(f"  no dev.json in the dataset — held out {n_dev} of "
                  f"{len(train)} training pairs as dev (seed {cfg.seed}). "
                  f"train is now {len(ti)} pairs, disjoint from dev.")
    return _SPLITS[key][0 if split == "train" else 1]


def save_notebook(dest_dir, stem="notebook"):
    """Write a copy of THIS notebook beside the run it produced.

    Colab exposes the live notebook through an internal message channel, which
    is the only reliable way to get it: a notebook opened from Drive is not at a
    predictable path, and one uploaded into the session is not in Drive at all."""
    dest = Path(dest_dir); dest.mkdir(parents=True, exist_ok=True)
    try:
        from google.colab import _message
        nb = _message.blocking_request("get_ipynb", request="", timeout_sec=120)["ipynb"]
        p = dest / f"{stem}.ipynb"
        p.write_text(json.dumps(nb, indent=1))
        print(f"  notebook copied -> {p}")
        return str(p)
    except Exception:
        pass
    print(f"  (could not capture the notebook automatically — save a copy into "
          f"{dest} by hand so the code sits with its results)")
    return None

### 2.3 The Phase 1 encoder–decoder

Copied from your `phase1_encdec.ipynb` so the `state_dict` loads into exactly the
code that produced it — same module names (`wq/wk/wv/wo`, `w1/w2/w3`, RMSNorm `.g`),
same interleaved RoPE, same weight tying.

**One addition:** an optional `key_padding_mask` on encoder self-attention and on
cross-attention. Your pretraining used fixed 512-token windows so padding never
existed; summarization has variable-length documents, and without a mask the
encoder would attend to `<pad>` — whose embedding is untrained noise. The mask adds
no parameters, so the checkpoint still loads strictly.

In [11]:
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__(); self.eps = eps; self.g = nn.Parameter(torch.ones(d))
    def forward(self, x):
        return self.g * x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)


def build_rope_cache(seq_len, head_dim, theta=10000.0, device="cpu"):
    inv = 1.0 / (theta ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(seq_len, device=device).float()
    fr = torch.outer(t, inv)
    return torch.cos(fr), torch.sin(fr)


def apply_rope(x, cos, sin):
    T = x.size(-2)
    cos = cos[:T].unsqueeze(0).unsqueeze(0)
    sin = sin[:T].unsqueeze(0).unsqueeze(0)
    x1 = x[..., 0::2]; x2 = x[..., 1::2]
    r1 = x1 * cos - x2 * sin
    r2 = x1 * sin + x2 * cos
    return torch.stack([r1, r2], dim=-1).flatten(-2)


def _kpm_to_attnmask(kpm, n_q, T):
    """[B, S] bool (True = PAD) -> [B, n_q, T, S] bool (True = attend)."""
    if kpm is None:
        return None
    return (~kpm)[:, None, None, :].expand(-1, n_q, T, -1)


class GQASelfAttention(nn.Module):
    def __init__(self, cfg, is_causal=True):
        super().__init__()
        self.n_q = cfg.n_heads; self.n_kv = cfg.n_kv_heads
        self.head_dim = cfg.head_dim; self.n_rep = self.n_q // self.n_kv
        self.is_causal = is_causal
        self.wq = nn.Linear(cfg.d_model, self.n_q * self.head_dim, bias=False)
        self.wk = nn.Linear(cfg.d_model, self.n_kv * self.head_dim, bias=False)
        self.wv = nn.Linear(cfg.d_model, self.n_kv * self.head_dim, bias=False)
        self.wo = nn.Linear(self.n_q * self.head_dim, cfg.d_model, bias=False)

    def forward(self, x, cos, sin, kv_cache=None, start_pos=0, kpm=None):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.n_q,  self.head_dim).transpose(1, 2)
        k = self.wk(x).view(B, T, self.n_kv, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(B, T, self.n_kv, self.head_dim).transpose(1, 2)
        q = apply_rope(q, cos[start_pos:start_pos + T], sin[start_pos:start_pos + T])
        k = apply_rope(k, cos[start_pos:start_pos + T], sin[start_pos:start_pos + T])
        if kv_cache is not None:
            pk, pv = kv_cache
            k = torch.cat([pk, k], dim=2); v = torch.cat([pv, v], dim=2)
        new_cache = (k, v)
        k_e = k.repeat_interleave(self.n_rep, dim=1) if self.n_rep > 1 else k
        v_e = v.repeat_interleave(self.n_rep, dim=1) if self.n_rep > 1 else v
        causal = self.is_causal and (kv_cache is None)
        mask = _kpm_to_attnmask(kpm, self.n_q, T)
        # SDPA takes is_causal OR an explicit mask, never both
        out = F.scaled_dot_product_attention(q, k_e, v_e, attn_mask=mask,
                                             is_causal=(causal and mask is None))
        return self.wo(out.transpose(1, 2).contiguous().view(B, T, -1)), new_cache


class GQACrossAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_q = cfg.n_heads; self.n_kv = cfg.n_kv_heads
        self.head_dim = cfg.head_dim; self.n_rep = self.n_q // self.n_kv
        self.wq = nn.Linear(cfg.d_model, self.n_q * self.head_dim, bias=False)
        self.wk = nn.Linear(cfg.d_model, self.n_kv * self.head_dim, bias=False)
        self.wv = nn.Linear(cfg.d_model, self.n_kv * self.head_dim, bias=False)
        self.wo = nn.Linear(self.n_q * self.head_dim, cfg.d_model, bias=False)

    def forward(self, x, memory, kv_cache=None, kpm=None):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.n_q, self.head_dim).transpose(1, 2)
        if kv_cache is not None:
            k, v = kv_cache
        else:
            S = memory.size(1)
            k = self.wk(memory).view(B, S, self.n_kv, self.head_dim).transpose(1, 2)
            v = self.wv(memory).view(B, S, self.n_kv, self.head_dim).transpose(1, 2)
        new_cache = (k, v)
        k_e = k.repeat_interleave(self.n_rep, dim=1) if self.n_rep > 1 else k
        v_e = v.repeat_interleave(self.n_rep, dim=1) if self.n_rep > 1 else v
        out = F.scaled_dot_product_attention(
            q, k_e, v_e, attn_mask=_kpm_to_attnmask(kpm, self.n_q, T),
            is_causal=False)
        return self.wo(out.transpose(1, 2).contiguous().view(B, T, -1)), new_cache


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)     # gate
        self.w2 = nn.Linear(d_ff, d_model, bias=False)     # down
        self.w3 = nn.Linear(d_model, d_ff, bias=False)     # up
    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n1 = RMSNorm(cfg.d_model, cfg.rms_eps)
        self.attn = GQASelfAttention(cfg, is_causal=False)
        self.n2 = RMSNorm(cfg.d_model, cfg.rms_eps)
        self.ff = SwiGLU(cfg.d_model, cfg.d_ff)
    def forward(self, x, cos, sin, kpm=None):
        a, _ = self.attn(self.n1(x), cos, sin, kpm=kpm)
        x = x + a
        return x + self.ff(self.n2(x))


class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n1 = RMSNorm(cfg.d_model, cfg.rms_eps)
        self.self_attn = GQASelfAttention(cfg, is_causal=True)
        self.n2 = RMSNorm(cfg.d_model, cfg.rms_eps)
        self.cross_attn = GQACrossAttention(cfg)
        self.n3 = RMSNorm(cfg.d_model, cfg.rms_eps)
        self.ff = SwiGLU(cfg.d_model, cfg.d_ff)
    def forward(self, x, memory, cos, sin, self_kv=None, cross_kv=None,
                start_pos=0, src_kpm=None):
        # decoder self-attention needs no padding mask: padding sits at the END of
        # the target and causal masking already stops earlier positions seeing it,
        # while the padded positions themselves are dropped by the loss
        a, nsk = self.self_attn(self.n1(x), cos, sin, self_kv, start_pos)
        x = x + a
        c, nck = self.cross_attn(self.n2(x), memory, cross_kv, kpm=src_kpm)
        x = x + c
        return x + self.ff(self.n3(x)), nsk, nck


class EncDecGQA(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        V = cfg.vocab_size
        self.src_emb = nn.Embedding(V, cfg.d_model)
        self.tgt_emb = self.src_emb
        self.enc_blocks = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.enc_layers)])
        self.dec_blocks = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.dec_layers)])
        self.enc_norm = RMSNorm(cfg.d_model, cfg.rms_eps)
        self.dec_norm = RMSNorm(cfg.d_model, cfg.rms_eps)
        self.head = nn.Linear(cfg.d_model, V, bias=False)
        self.head.weight = self.src_emb.weight
        max_len = max(cfg.src_len, cfg.tgt_len) * 4
        cos, sin = build_rope_cache(max_len, cfg.head_dim, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def encode(self, src, src_kpm=None):
        x = self.src_emb(src)
        for blk in self.enc_blocks:
            x = blk(x, self.rope_cos, self.rope_sin, kpm=src_kpm)
        return self.enc_norm(x)

    def forward(self, src, tgt_in, src_kpm=None):
        memory = self.encode(src, src_kpm)
        x = self.tgt_emb(tgt_in)
        for blk in self.dec_blocks:
            x, _, _ = blk(x, memory, self.rope_cos, self.rope_sin,
                          src_kpm=src_kpm)
        return self.head(self.dec_norm(x))

    def decode_cached(self, tgt_in, memory, self_caches, cross_caches, start_pos,
                      src_kpm=None):
        x = self.tgt_emb(tgt_in)
        ns, nc = [], []
        for i, blk in enumerate(self.dec_blocks):
            x, a, b = blk(x, memory, self.rope_cos, self.rope_sin,
                          self_caches[i] if self_caches else None,
                          cross_caches[i] if cross_caches else None,
                          start_pos, src_kpm)
            ns.append(a); nc.append(b)
        return self.head(self.dec_norm(x)), ns, nc


def load_phase1(cfg, enc_keep=None, dec_keep=None, verbose=True):
    """Builds the summarizer on the Phase 1 weights. enc_keep/dec_keep take an
    evenly spaced subset of blocks — that is how the distillation student is made."""
    ck = torch.load(cfg.ckpt_p1, map_location="cpu", weights_only=False)
    sd = ck["model"] if "model" in ck else ck
    sd = {(k[7:] if k.startswith("module.") else k): v for k, v in sd.items()}
    saved = ck.get("cfg", {}) or {}

    V, d_model = sd["src_emb.weight"].shape
    enc_n = max(int(m.group(1)) for k in sd
                if (m := re.match(r"enc_blocks\.(\d+)\.", k))) + 1
    dec_n = max(int(m.group(1)) for k in sd
                if (m := re.match(r"dec_blocks\.(\d+)\.", k))) + 1
    d_ff = sd["enc_blocks.0.ff.w1.weight"].shape[0]
    n_heads = int(saved.get("n_heads", cfg.n_heads))
    head_dim = d_model // n_heads
    n_kv = sd["enc_blocks.0.attn.wk.weight"].shape[0] // head_dim

    bc = replace(cfg, d_model=d_model, vocab_size=V, d_ff=d_ff, n_heads=n_heads,
                 n_kv_heads=n_kv, enc_layers=enc_n, dec_layers=dec_n,
                 rope_theta=float(saved.get("rope_theta", cfg.rope_theta)),
                 rms_eps=float(saved.get("rms_eps", cfg.rms_eps)))
    ei = list(range(enc_n)); di = list(range(dec_n))
    if enc_keep and enc_keep < enc_n:
        ei = [min(enc_n - 1, int(round(i * enc_n / enc_keep))) for i in range(enc_keep)]
        bc = replace(bc, enc_layers=enc_keep)
    if dec_keep and dec_keep < dec_n:
        di = [min(dec_n - 1, int(round(i * dec_n / dec_keep))) for i in range(dec_keep)]
        bc = replace(bc, dec_layers=dec_keep)

    model = EncDecGQA(bc)
    want, new = model.state_dict(), {}
    for k in want:
        src = k
        if k.startswith("enc_blocks."):
            i = int(k.split(".")[1]); src = k.replace(f"enc_blocks.{i}.", f"enc_blocks.{ei[i]}.", 1)
        elif k.startswith("dec_blocks."):
            i = int(k.split(".")[1]); src = k.replace(f"dec_blocks.{i}.", f"dec_blocks.{di[i]}.", 1)
        if src in sd and sd[src].shape == want[k].shape:
            new[k] = sd[src]
    # Tied parameters (src_emb / tgt_emb / head all share one tensor) appear
    # under several state_dict names. A name that did not match is harmless as
    # long as one of its aliases did, so resolve aliases before complaining.
    _by = {}
    for k, v in want.items():
        _by.setdefault(id(v), []).append(k)
    _alias = {k: ks for ks in _by.values() for k in ks}
    missing = [k for k in want
               if k not in new and "rope_" not in k
               and not any(a in new for a in _alias[k])]
    model.load_state_dict(new, strict=False)

    if verbose:
        print(f"Phase 1 enc-dec: {d_model}d  enc={bc.enc_layers}L dec={bc.dec_layers}L  "
              f"q={n_heads} kv={n_kv}  d_ff={d_ff}  vocab={V:,}")
        print(f"  loaded {len(new)}/{len(want)} tensors from {Path(cfg.ckpt_p1).name} "
              f"(step {ck.get('step','?')}, val_loss {ck.get('best_val', float('nan')):.4f})")
        if ei != list(range(enc_n)) or di != list(range(dec_n)):
            print(f"  block subset: enc {ei}  dec {di}")
        if missing:
            print(f"  NOT loaded (fresh init): {missing[:6]}")
        std = model.src_emb.weight.std().item()
        print(f"  src_emb std = {std:.4f} (fresh init ~0.0200)")
        if abs(std - 0.02) < 0.002:
            print("  !! near init — check the checkpoint really loaded")
    return model, bc

### 2.4 PEFT, written from scratch

The same methods as Assignment 4 Q4 / Q5.1 and Phase 5, retargeted to this
architecture's module names. No `peft`, no `bitsandbytes`.

An encoder–decoder has **three** attention sites, not one: encoder self-attention,
decoder self-attention and **cross-attention**. Cross-attention is where the summary
reads the document, so it is the interesting place to adapt — `TARGET_SITES` lets
each experiment choose, and that choice is itself a finding worth reporting.

In [12]:
ATTN_T = ("wq", "wk", "wv", "wo")
FFN_T  = ("w1", "w2", "w3")


def param_report(model, label=""):
    tot = sum(p.numel() for p in model.parameters())
    tot += sum(m.n for m in model.modules() if m.__class__.__name__ == "NF4Linear")
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{label}trainable {tr:,} / {tot:,} = {100*tr/max(tot,1):.3f}%")
    return tr, tot


def freeze_all(m):
    for p in m.parameters():
        p.requires_grad = False


def attn_sites(model, sites=("enc", "dec_self", "cross")):
    """Yields the attention modules at the requested sites."""
    out = []
    if "enc" in sites:
        out += [b.attn for b in model.enc_blocks]
    if "dec_self" in sites:
        out += [b.self_attn for b in model.dec_blocks]
    if "cross" in sites:
        out += [b.cross_attn for b in model.dec_blocks]
    return out


def ffn_sites(model, sites=("enc", "dec")):
    out = []
    if "enc" in sites:
        out += [b.ff for b in model.enc_blocks]
    if "dec" in sites:
        out += [b.ff for b in model.dec_blocks]
    return out


class LoRALinear(nn.Module):
    """h = W0 x + (alpha/r) B A x ;  B = 0 at init so the model starts unchanged."""
    def __init__(self, base, r=16, alpha=32, p_drop=0.0, swap_ab=False):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        self.r, self.alpha, self.scaling = r, alpha, alpha / r
        self.lora_A = nn.Parameter(torch.zeros(r, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.lora_B if swap_ab else self.lora_A, a=math.sqrt(5))
        self.drop = nn.Dropout(p_drop)
    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.lora_A.T) @ self.lora_B.T * self.scaling
    @torch.no_grad()
    def merged_copy(self):
        W = (self.base.weight if hasattr(self.base, "weight")
             else self.base._deq())          # NF4 base: dequantize to merge
        new = nn.Linear(self.base.in_features, self.base.out_features, bias=False)
        new.weight.copy_(W + self.scaling * (self.lora_B @ self.lora_A))
        return new


LORMA_VARIANT = {"name": "LoRMA+",
                 "equation": "h = ((alpha/r) B A + I) W0 x",
                 "paper": "Bihany, Patel & Modi, arXiv:2506.07621v1, Eq. 8"}


class LoRMALinear(nn.Module):
    """Multiplicative: W' = ((alpha/r) B A + I) W0, applied in re-ordered form so
    the d_out x d_out product is never materialised."""
    def __init__(self, base, r=16, alpha=32, p_drop=0.0, swap_ab=False):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        d_out = base.out_features
        self.r, self.alpha, self.scaling = r, alpha, alpha / r
        self.lorma_A = nn.Parameter(torch.zeros(r, d_out))
        self.lorma_B = nn.Parameter(torch.zeros(d_out, r))
        nn.init.kaiming_uniform_(self.lorma_B if swap_ab else self.lorma_A, a=math.sqrt(5))
        self.drop = nn.Dropout(p_drop)
    def forward(self, x):
        y = self.base(x)
        return y + (self.drop(y) @ self.lorma_A.T) @ self.lorma_B.T * self.scaling


class BottleneckAdapter(nn.Module):
    """h' = h + W_up GELU(W_down RMSNorm(h)), W_up zero-init -> identity at step 0."""
    def __init__(self, d_model, bottleneck=64, eps=1e-5):
        super().__init__()
        self.norm = RMSNorm(d_model, eps)
        self.down = nn.Linear(d_model, bottleneck)
        self.up = nn.Linear(bottleneck, d_model)
        nn.init.normal_(self.down.weight, std=0.02); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
        self.enabled = True
    def forward(self, h):
        return h if not self.enabled else h + self.up(F.gelu(self.down(self.norm(h))))


class AdaptedEncoderBlock(nn.Module):
    """Houlsby: an adapter after EVERY sublayer. Pfeiffer: one, after the FFN.
    The two placements are listed as separate techniques in the task statement,
    and they genuinely differ — Houlsby can correct the attention output before
    the FFN ever sees it, Pfeiffer can only correct what comes out of the block."""
    def __init__(self, blk, d_model, bottleneck=64, variant="houlsby"):
        super().__init__()
        self.block = blk; self.variant = variant
        self.adapter_ffn = BottleneckAdapter(d_model, bottleneck)
        self.adapter_attn = (BottleneckAdapter(d_model, bottleneck)
                             if variant == "houlsby" else None)
    def forward(self, x, cos, sin, kpm=None):
        b = self.block
        a, _ = b.attn(b.n1(x), cos, sin, kpm=kpm)
        x = x + (self.adapter_attn(a) if self.adapter_attn is not None else a)
        return x + self.adapter_ffn(b.ff(b.n2(x)))


class AdaptedDecoderBlock(nn.Module):
    def __init__(self, blk, d_model, bottleneck=64, variant="houlsby"):
        super().__init__()
        self.block = blk; self.variant = variant
        self.adapter_ffn = BottleneckAdapter(d_model, bottleneck)
        if variant == "houlsby":
            self.adapter_self = BottleneckAdapter(d_model, bottleneck)
            self.adapter_cross = BottleneckAdapter(d_model, bottleneck)
        else:
            self.adapter_self = self.adapter_cross = None
    def forward(self, x, memory, cos, sin, self_kv=None, cross_kv=None,
                start_pos=0, src_kpm=None):
        b = self.block
        a, nsk = b.self_attn(b.n1(x), cos, sin, self_kv, start_pos)
        x = x + (self.adapter_self(a) if self.adapter_self is not None else a)
        c, nck = b.cross_attn(b.n2(x), memory, cross_kv, kpm=src_kpm)
        x = x + (self.adapter_cross(c) if self.adapter_cross is not None else c)
        return x + self.adapter_ffn(b.ff(b.n3(x))), nsk, nck


class SoftPromptEncDec(nn.Module):
    """m trainable vectors prepended to the ENCODER input. The decoder reads them
    through cross-attention, so the prompt conditions the whole summary. The
    backbone is frozen but differentiable — the gradient reaches the prompt only
    by passing through the encoder."""
    def __init__(self, model, n_virtual=32, init_ids=None):
        super().__init__()
        self.model = model
        emb = model.src_emb.weight
        if init_ids is None:
            start = min(1000, emb.size(0) - n_virtual - 1)
            init_ids = list(range(start, start + n_virtual))
        self.prompt = nn.Parameter(emb[torch.tensor(init_ids)].detach().clone().float())
    @property
    def src_emb(self):
        return self.model.src_emb
    def _mem(self, src, src_kpm):
        m = self.model
        x = m.src_emb(src)
        B = x.size(0); p = self.prompt.unsqueeze(0).expand(B, -1, -1).to(x.dtype)
        x = torch.cat([p, x], 1)
        if src_kpm is not None:
            src_kpm = torch.cat(
                [torch.zeros(B, self.prompt.size(0), dtype=src_kpm.dtype,
                             device=src_kpm.device), src_kpm], 1)
        for blk in m.enc_blocks:
            x = blk(x, m.rope_cos, m.rope_sin, kpm=src_kpm)
        return m.enc_norm(x), src_kpm
    def forward(self, src, tgt_in, src_kpm=None):
        m = self.model
        memory, kpm = self._mem(src, src_kpm)
        x = m.tgt_emb(tgt_in)
        for blk in m.dec_blocks:
            x, _, _ = blk(x, memory, m.rope_cos, m.rope_sin, src_kpm=kpm)
        return m.head(m.dec_norm(x))
    def encode(self, src, src_kpm=None):
        return self._mem(src, src_kpm)[0]
    def decode_cached(self, *a, **kw):
        return self.model.decode_cached(*a, **kw)
    @property
    def rope_cos(self): return self.model.rope_cos
    @property
    def rope_sin(self): return self.model.rope_sin


NF4_LEVELS = torch.tensor([
    -1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,
    -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,
    0.07958029955625534, 0.16093020141124725, 0.24611230194568634,
    0.33791524171829224, 0.44070982933044434, 0.5626170039176941,
    0.7229568362236023, 1.0], dtype=torch.float32)


class NF4Linear(nn.Module):
    """Frozen weight stored in 4 bits (blockwise absmax, two indices per byte),
    dequantized inside the forward. No trainable parameters — QLoRA is this plus a
    full-precision LoRA branch."""
    def __init__(self, base, block=64):
        super().__init__()
        W = base.weight.detach().float().reshape(-1)
        n = W.numel(); pad = (-n) % block
        if pad:
            W = torch.cat([W, torch.zeros(pad)])
        blocks = W.view(-1, block)
        scales = blocks.abs().amax(1).clamp(min=1e-12)
        idx = ((blocks / scales[:, None])[..., None] - NF4_LEVELS).abs().argmin(-1)
        idx = idx.reshape(-1).to(torch.uint8)
        self.register_buffer("packed", (idx[0::2] << 4) | idx[1::2])
        self.register_buffer("scales", scales.half())
        self.shape = tuple(base.weight.shape); self.n = n; self.block = block
        self.in_features, self.out_features = base.in_features, base.out_features
        with torch.no_grad():
            self.rel_err = float((self._deq() - base.weight).norm() / base.weight.norm())
    def _deq(self, dtype=torch.float32):
        hi = (self.packed >> 4).long(); lo = (self.packed & 0x0F).long()
        v = NF4_LEVELS.to(self.packed.device)[torch.stack([hi, lo], 1).reshape(-1)]
        out = (v.view(-1, self.block) * self.scales.float()[:, None]).reshape(-1)[:self.n]
        return out.view(self.shape).to(dtype)
    def stored_bytes(self):
        return self.packed.numel() + self.scales.numel() * 2
    def forward(self, x):
        return F.linear(x, self._deq(x.dtype), None)


def _wrappable(base):
    """A projection we can put a low-rank branch around. This must accept
    NF4Linear as well as nn.Linear: in QLoRA the base is quantized FIRST, so by
    the time the branch is attached the module is no longer an nn.Linear — an
    isinstance(nn.Linear) test there silently attaches nothing and the run dies
    with 'nothing trainable'."""
    return (hasattr(base, "in_features") and hasattr(base, "out_features")
            and not isinstance(base, (LoRALinear, LoRMALinear)))


def inject_lowrank(mods, cls, r=16, alpha=32, targets=ATTN_T, swap_ab=False):
    out = []
    for m in mods:
        for name in targets:
            base = getattr(m, name, None)
            if _wrappable(base):
                new = cls(base, r, alpha, 0.0, swap_ab)
                setattr(m, name, new); out.append(new)
    if not out:
        raise RuntimeError("inject_lowrank matched no projections — check the "
                           "target names against this architecture")
    return out


def inject_adapters(model, bottleneck=64, sites=("enc", "dec"),
                    variant="houlsby"):
    d = model.cfg.d_model if hasattr(model, "cfg") else model.src_emb.weight.size(1)
    ads = []
    if "enc" in sites:
        model.enc_blocks = nn.ModuleList(
            [AdaptedEncoderBlock(b, d, bottleneck, variant)
             for b in model.enc_blocks])
    if "dec" in sites:
        model.dec_blocks = nn.ModuleList(
            [AdaptedDecoderBlock(b, d, bottleneck, variant)
             for b in model.dec_blocks])
    for blk in list(model.enc_blocks) + list(model.dec_blocks):
        ads += [m for m in blk.children() if isinstance(m, BottleneckAdapter)]
    return ads


def quantize_nf4(model, block=64):
    q, fp, qb, errs = [], 0, 0, []
    for m in attn_sites(model) + ffn_sites(model):
        for name in (ATTN_T + FFN_T):
            base = getattr(m, name, None)
            if isinstance(base, nn.Linear):
                fp += base.weight.numel() * 4
                new = NF4Linear(base, block)
                qb += new.stored_bytes(); errs.append(new.rel_err)
                setattr(m, name, new); q.append(new)
    print(f"NF4: {len(q)} projections  {fp/1e6:.0f} MB fp32 -> {qb/1e6:.0f} MB "
          f"({fp/max(qb,1):.2f}x smaller), mean relative error {np.mean(errs):.4f}")
    return q, {"fp32_mb": fp/1e6, "nf4_mb": qb/1e6, "ratio": fp/max(qb,1),
               "rel_err": float(np.mean(errs))}

### 2.5 Training, generation, metrics, submission

Two decisions made for the deadline, both worth stating in the report:

* **Model selection during training uses dev *loss*, not ROUGE.** Generating a few
  hundred summaries at every evaluation would cost more than the training does.
  ROUGE is computed once, at the end, on a dev subset.
* **Every run is capped by wall clock** (`max_minutes`). The LR schedule is built
  from the step cap so the cosine still completes; a run that hits the time cap
  stops cleanly on its best checkpoint rather than being killed.

In [13]:
def masked_ce(logits, target, label_smoothing=0.0):
    """Cross-entropy over real target positions only. Phase 1's forward() used a
    plain cross_entropy because its windows were never padded; a summary is."""
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(),
                           target.reshape(-1), ignore_index=PAD_ID,
                           label_smoothing=label_smoothing)


@torch.no_grad()
def eval_loss(model, loader, cfg, max_batches=60):
    model.eval(); tot = n = 0
    for bi, (s, ti, to, _) in enumerate(loader):
        if bi >= max_batches:
            break
        s, ti, to = s.to(DEVICE), ti.to(DEVICE), to.to(DEVICE)
        with torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            logits = model(s, ti, src_kpm=(s == PAD_ID))
        tot += masked_ce(logits, to).item(); n += 1
    model.train()
    return tot / max(n, 1)


def train(model, train_loader, dev_loader, cfg, loss_fn=None, label="run",
          start_step=0):
    model.to(DEVICE)
    groups = [p for p in model.parameters() if p.requires_grad]
    if not groups:
        raise RuntimeError("nothing trainable")
    decay = [p for p in groups if p.ndim >= 2]
    nodecay = [p for p in groups if p.ndim < 2]
    opt = torch.optim.AdamW(
        [{"params": decay, "weight_decay": cfg.weight_decay},
         {"params": nodecay, "weight_decay": 0.0}], lr=cfg.lr, betas=(0.9, 0.95))
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

    spe = max(1, math.ceil(len(train_loader) / cfg.grad_accum))
    total = spe * cfg.epochs
    if cfg.max_steps:
        total = min(total, cfg.max_steps)
    if start_step >= total:
        print(f"  [{label}] already at step {start_step}/{total} — raise "
              f"cfg.epochs or cfg.max_steps to train further")
        return {"best_val_loss": eval_loss(model, dev_loader, cfg),
                "history": [], "steps": start_step, "train_seconds": 0.0,
                "peak_mem_mb": 0.0}
    warm = max(1, int(cfg.warmup_frac * total))
    base_lr = cfg.lr
    deadline = time.time() + cfg.max_minutes * 60 if cfg.max_minutes else None

    def lr_at(s):
        if s < warm:
            return (s + 1) / warm
        p = (s - warm) / max(1, total - warm)
        return 0.5 * (1 + math.cos(math.pi * min(p, 1.0)))

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    hist, best, best_sd, no_imp, micro = [], float("inf"), None, 0, 0
    step = start_step
    if start_step:
        print(f"  [{label}] resuming at step {start_step}/{total} "
              f"(the LR schedule continues from there)")
    t0 = time.time(); model.train(); stop = False
    for ep in range(cfg.epochs):
        if stop:
            break
        for s, ti, to, _ in train_loader:
            s, ti, to = s.to(DEVICE), ti.to(DEVICE), to.to(DEVICE)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
                logits = model(s, ti, src_kpm=(s == PAD_ID))
                loss = (loss_fn(logits, to, s) if loss_fn
                        else masked_ce(logits, to, cfg.label_smoothing))
                loss = loss / cfg.grad_accum
            scaler.scale(loss).backward()
            micro += 1
            if micro % cfg.grad_accum:
                continue
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(groups, cfg.max_grad_norm)
            f = lr_at(step)
            for g in opt.param_groups:
                g["lr"] = base_lr * f
            scaler.step(opt); scaler.update()
            opt.zero_grad(set_to_none=True)
            step += 1

            if step % cfg.eval_every == 0 or step == total:
                vl = eval_loss(model, dev_loader, cfg)
                hist.append({"step": step, "val_loss": vl,
                             "ppl": math.exp(min(vl, 20))})
                flag = ""
                if vl < best:
                    best, no_imp = vl, 0
                    best_sd = {k: v.detach().cpu().clone()
                               for k, v in model.state_dict().items()}
                    flag = "  *best*"
                else:
                    no_imp += 1
                el = (time.time() - t0) / 60
                print(f"  [{label}] step {step}/{total} val_loss {vl:.4f} "
                      f"ppl {math.exp(min(vl,20)):.1f}{flag}  ({el:.1f} min)")
                model.train()
                if no_imp >= cfg.patience:
                    print(f"  [{label}] early stop"); stop = True; break
            if deadline and time.time() > deadline:
                print(f"  [{label}] TIME CAP reached at step {step} "
                      f"({cfg.max_minutes} min) — keeping the best checkpoint")
                stop = True; break
            if step >= total:
                stop = True; break

    if best_sd is None:                      # cap hit before the first eval
        vl = eval_loss(model, dev_loader, cfg)
        best = vl
        best_sd = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        hist.append({"step": step, "val_loss": vl, "ppl": math.exp(min(vl, 20))})
    model.load_state_dict(best_sd)
    return {"best_val_loss": best, "history": hist, "steps": step,
            "start_step": start_step,
            "train_seconds": time.time() - t0,
            "peak_mem_mb": (torch.cuda.max_memory_allocated() / 1e6
                            if DEVICE.type == "cuda" else 0.0)}


# ════════════════════════════════════════════════ generation

def _block_ngrams(seqs, logits, n):
    """no-repeat-n-gram: forbid any token that would complete an n-gram already
    produced. Abstractive decoders loop without this, and a looping summary is
    punished hard by ROUGE."""
    if n <= 0:
        return
    for b, seq in enumerate(seqs):
        if len(seq) < n - 1:
            continue
        prefix = tuple(seq[-(n - 1):])
        seen = {seq[i + n - 1] for i in range(len(seq) - n + 1)
                if tuple(seq[i:i + n - 1]) == prefix}
        for t in seen:
            logits[b, t] = -float("inf")


def _encode_for_gen(model, src, kpm):
    if hasattr(model, "_mem"):              # soft-prompt wrapper extends the mask
        return model._mem(src, kpm)
    return model.encode(src, kpm), kpm


@torch.no_grad()
def generate_beam(model, S, cfg, batch_size=4, beams=4, desc=""):
    """Beam search with KV cache, length penalty and no-repeat-n-gram.

    Greedy commits to a token it cannot take back, which on summarization tends to
    start a clause it then has to pad out. Beam search keeps `beams` hypotheses and
    scores whole sequences, normalised by length^gen_len_penalty so it does not
    simply prefer the shortest summary."""
    model.to(DEVICE).eval()
    K, outs, t_start = beams, [], time.time()
    for i in range(0, len(S), batch_size):
        src = torch.from_numpy(S[i:i + batch_size].astype(np.int64)).to(DEVICE)
        kpm = (src == PAD_ID); B = src.size(0)
        with torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            memory, kpm2 = _encode_for_gen(model, src, kpm)
        Sm, D = memory.size(1), memory.size(2)
        memory = memory.unsqueeze(1).expand(B, K, Sm, D).reshape(B * K, Sm, D)
        kpm2 = (None if kpm2 is None else
                kpm2.unsqueeze(1).expand(B, K, -1).reshape(B * K, -1))
        tok = torch.full((B * K, 1), BOS_ID, dtype=torch.long, device=DEVICE)
        sc = cc = None
        scores = torch.full((B, K), -1e9, device=DEVICE); scores[:, 0] = 0.0
        seqs = [[] for _ in range(B * K)]
        fin = [[] for _ in range(B)]
        for t in range(cfg.gen_max_new):
            with torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
                logits, sc, cc = model.decode_cached(tok, memory, sc, cc,
                                                     start_pos=t, src_kpm=kpm2)
            lp = F.log_softmax(logits[:, -1].float(), dim=-1)
            lp[:, PAD_ID] = -1e9; lp[:, BOS_ID] = -1e9
            if t < cfg.gen_min_new:
                lp[:, EOS_ID] = -1e9
            _block_ngrams(seqs, lp, cfg.gen_no_repeat_ngram)
            V = lp.size(-1)
            cand = (scores.view(B * K, 1) + lp).view(B, K * V)
            top_s, top_i = cand.topk(K, dim=-1)
            beam_idx = torch.div(top_i, V, rounding_mode="floor")
            tok_idx = top_i % V
            flat = (torch.arange(B, device=DEVICE).unsqueeze(1) * K
                    + beam_idx).view(-1)
            sc = [(k[flat], v[flat]) for k, v in sc]
            cc = [(k[flat], v[flat]) for k, v in cc]
            seqs = [seqs[b * K + int(beam_idx[b, j])] + [int(tok_idx[b, j])]
                    for b in range(B) for j in range(K)]
            scores = top_s.clone()
            for b in range(B):
                for j in range(K):
                    if int(tok_idx[b, j]) == EOS_ID:
                        toks = seqs[b * K + j][:-1]
                        pen = max(len(toks), 1) ** cfg.gen_len_penalty
                        fin[b].append((float(scores[b, j]) / pen, toks))
                        scores[b, j] = -1e9
            if all(len(f) >= K for f in fin):
                break
            tok = tok_idx.reshape(B * K, 1)
        for b in range(B):
            if fin[b]:
                outs.append(max(fin[b], key=lambda x: x[0])[1])
            else:
                outs.append(seqs[b * K + int(scores[b].argmax())])
        if desc and (i // max(batch_size, 1)) % 10 == 0:
            print(f"  {desc} {min(i+batch_size, len(S))}/{len(S)} "
                  f"({time.time()-t_start:.0f}s)")
    if desc:
        print(f"  {desc} {len(S)} summaries (beam {K}) in "
              f"{time.time()-t_start:.0f}s "
              f"(mean {np.mean([len(s) for s in outs]):.0f} tokens)")
    return outs


@torch.no_grad()
def generate(model, S, cfg, batch_size=8, desc="", beams=None):
    """Greedy by default; dispatches to beam search when cfg.gen_beams > 1."""
    K = cfg.gen_beams if beams is None else beams
    if K and K > 1:
        return generate_beam(model, S, cfg, max(1, batch_size // K), K, desc)
    model.to(DEVICE).eval()
    outs, t0 = [], time.time()
    for i in range(0, len(S), batch_size):
        src = torch.from_numpy(S[i:i + batch_size].astype(np.int64)).to(DEVICE)
        kpm = (src == PAD_ID)
        B = src.size(0)
        with torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            memory, kpm2 = _encode_for_gen(model, src, kpm)
        tok = torch.full((B, 1), BOS_ID, dtype=torch.long, device=DEVICE)
        sc = cc = None
        seqs = [[] for _ in range(B)]
        done = torch.zeros(B, dtype=torch.bool, device=DEVICE)
        for t in range(cfg.gen_max_new):
            with torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
                logits, sc, cc = model.decode_cached(tok, memory, sc, cc,
                                                     start_pos=t, src_kpm=kpm2)
            lg = logits[:, -1].float()
            lg[:, PAD_ID] = -float("inf")
            lg[:, BOS_ID] = -float("inf")
            if t < cfg.gen_min_new:
                lg[:, EOS_ID] = -float("inf")          # enforce a minimum length
            _block_ngrams(seqs, lg, cfg.gen_no_repeat_ngram)
            nxt = lg.argmax(-1)
            nxt = torch.where(done, torch.full_like(nxt, EOS_ID), nxt)
            for b in range(B):
                if not done[b] and nxt[b].item() != EOS_ID:
                    seqs[b].append(int(nxt[b].item()))
            done = done | (nxt == EOS_ID)
            if bool(done.all()):
                break
            tok = nxt.unsqueeze(1)
        outs.extend(seqs)
        if desc and (i // max(batch_size, 1)) % 20 == 0:
            print(f"  {desc} {min(i+batch_size, len(S))}/{len(S)} "
                  f"({time.time()-t0:.0f}s)")
    if desc:
        print(f"  {desc} {len(S)} summaries in {time.time()-t0:.0f}s "
              f"(mean {np.mean([len(s) for s in outs]):.0f} tokens)")
    return outs


# ════════════════════════════════════════════════ metrics

def _norm(s):
    return re.sub(r"[^a-z0-9 ]+", " ", str(s).lower()).split()


def rouge_n(ref, cand, n=1):
    r, c = _norm(ref), _norm(cand)
    if not r or not c:
        return 0.0
    rg = [tuple(r[i:i+n]) for i in range(len(r)-n+1)]
    cg = [tuple(c[i:i+n]) for i in range(len(c)-n+1)]
    if not rg or not cg:
        return 0.0
    from collections import Counter
    overlap = sum((Counter(rg) & Counter(cg)).values())
    p = overlap / len(cg); rc = overlap / len(rg)
    return 0.0 if p + rc == 0 else 2 * p * rc / (p + rc)


def rouge_l(ref, cand):
    r, c = _norm(ref), _norm(cand)
    if not r or not c:
        return 0.0
    prev = [0] * (len(c) + 1)
    for i in range(1, len(r) + 1):
        cur = [0] * (len(c) + 1)
        ri = r[i-1]
        for j in range(1, len(c) + 1):
            cur[j] = prev[j-1] + 1 if ri == c[j-1] else max(prev[j], cur[j-1])
        prev = cur
    lcs = prev[-1]
    p = lcs / len(c); rc = lcs / len(r)
    return 0.0 if p + rc == 0 else 2 * p * rc / (p + rc)


def score_summaries(refs, cands):
    keep = [(r, c) for r, c in zip(refs, cands) if r]
    if not keep:
        return {}
    r1 = float(np.mean([rouge_n(r, c, 1) for r, c in keep]))
    r2 = float(np.mean([rouge_n(r, c, 2) for r, c in keep]))
    rl = float(np.mean([rouge_l(r, c) for r, c in keep]))
    lens = [len(_norm(c)) for _, c in keep]
    rlens = [len(_norm(r)) for r, _ in keep]
    # redundancy: share of generated unigrams that are repeats
    rep = float(np.mean([1 - len(set(_norm(c))) / max(len(_norm(c)), 1)
                         for _, c in keep]))
    return {"rouge1": r1, "rouge2": r2, "rougeL": rl,
            "gen_len": float(np.mean(lens)), "ref_len": float(np.mean(rlens)),
            "len_ratio": float(np.mean(lens) / max(np.mean(rlens), 1)),
            "repetition": rep, "n": len(keep)}


# ════════════════════════════════════════════════ results + submission

def load_results(cfg):
    p = Path(cfg.results_json)
    return json.loads(p.read_text()) if p.exists() else []


def find_prior(cfg, name):
    """A previous checkpoint for this experiment: in the output folder first,
    then anywhere under Drive. Returns the
    loaded checkpoint, or None."""
    cands = [Path(cfg.out_dir) / "models" / f"{name}.pt"]
    cands += sorted(DRIVE.glob(f"**/cs781_phase6/models/{name}.pt"))
    for p in cands:
        if p.exists():
            try:
                return torch.load(p, map_location="cpu", weights_only=False), p
            except Exception as e:
                print(f"  (could not read {p}: {type(e).__name__})")
    return None, None


def planned_steps(cfg, train_loader):
    spe = max(1, math.ceil(len(train_loader) / cfg.grad_accum))
    t = spe * cfg.epochs
    return min(t, cfg.max_steps) if cfg.max_steps else t


def resume_into(model, cfg, name, train_loader, peft_injected=True):
    """Load a previous run of this experiment into `model` and say which step to
    continue from. Call AFTER any PEFT modules are injected, so the structure
    matches. Returns (start_step, skip) — skip is True when the previous run
    already reached the planned budget, so there is nothing to add."""
    if not cfg.resume:
        return 0, False
    ck, p = find_prior(cfg, name)
    if ck is None:
        return 0, False
    done = int(ck.get("trained_steps", 0))
    target = planned_steps(cfg, train_loader)
    miss, unexp = model.load_state_dict(ck["model"], strict=False)
    miss = [m for m in miss if "rope_" not in m]
    if miss or unexp:
        print(f"  !! {p.name} does not match this model "
              f"(missing {miss[:3]} unexpected {unexp[:3]}) — starting fresh")
        return 0, False
    print(f"  resumed {name} from {p} at step {done}/{target} "
          f"(val_loss {ck.get('best_val_loss', float('nan')):.4f})")
    if done >= target:
        print(f"  nothing to add at epochs={cfg.epochs}. Raise cfg.epochs or "
              f"cfg.max_steps and re-run to train further.")
        return done, True
    return done, False


def already_done(cfg, name):
    rows = load_results(cfg)
    ck = Path(cfg.out_dir) / "models" / f"{name}.pt"
    if any(r.get("name") == name for r in rows) and ck.exists():
        r = next(r for r in rows if r["name"] == name)
        print(f"SKIP {name} — on disk (dev loss {r['best_val_loss']:.4f}, "
              f"ROUGE-L {r.get('rougeL', float('nan')):.4f})")
        return True
    return False


def save_run(cfg, name, model, bc, technique, peft, info, metrics, tr, tot, extra=None):
    p = Path(cfg.out_dir) / "models" / f"{name}.pt"
    ck = {"model": {k: v.detach().cpu() for k, v in model.state_dict().items()},
          "arch": {k: v for k, v in asdict(bc).items()
                   if isinstance(v, (int, float, str, bool, type(None)))},
          "name": name, "technique": technique, "peft": peft,
          "src_len": cfg.src_len, "tgt_len": cfg.tgt_len,
          "doc_strategy": cfg.doc_strategy,
          "best_val_loss": info["best_val_loss"], "history": info["history"],
          "trained_steps": info["steps"], "epochs_cfg": cfg.epochs,
          "trainable": int(tr), "total": int(tot), "metrics": metrics}
    if extra:
        ck.update(extra)
    torch.save(ck, p)
    print(f"saved {p} ({p.stat().st_size/1e6:.0f} MB)")
    return str(p)


def record(cfg, name, technique, info, metrics, tr, tot, costs=None, extra=None):
    p = Path(cfg.results_json)
    rows = [r for r in load_results(cfg) if r.get("name") != name]
    row = {"name": name, "technique": technique,
           "best_val_loss": info["best_val_loss"],
           "ppl": math.exp(min(info["best_val_loss"], 20)),
           "trainable_params": int(tr), "total_params": int(tot),
           "trainable_pct": round(100 * tr / max(tot, 1), 4),
           "train_minutes": round(info["train_seconds"] / 60, 1),
           "steps": info["steps"], "peak_mem_mb": round(info.get("peak_mem_mb", 0), 1),
           "history": info["history"], "src_len": cfg.src_len,
           "tgt_len": cfg.tgt_len, "doc_strategy": cfg.doc_strategy,
           "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
    row.update(metrics or {})
    row.update(costs or {})
    if extra:
        row.update(extra)
    rows.append(row)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(rows, indent=1, default=float))
    print(f"recorded '{name}'")
    return row


def write_submission(cfg, model, name, tok, test_rows=None, batch_size=8):
    """submissions/<name>/prediction.json = [{"id":..., "summary":...}] + zip.
    List format, which the task statement recommends."""
    if test_rows is None:
        test_rows = load_split(cfg, "test")
    print(f"test documents: {len(test_rows)}")
    tag = f"test_{cfg.doc_strategy}_{cfg.src_len}"
    f = Path(cfg.cache_dir) / f"p6_{tag}.npz"
    if f.exists():
        S = np.load(f)["S"]
        print(f"[cache hit ] test source ({f.name})")
    else:
        S, _, _, _, _ = encode_pairs(tok, [(i, d, None) for i, d, _ in test_rows],
                                     cfg, desc="test")
        np.savez_compressed(f, S=S)
    ids = generate(model, S, cfg, batch_size=batch_size, desc="generating")
    sub = []
    for k, (rid, _, _) in enumerate(test_rows):
        txt = dec_ids(tok, ids[k]).strip()
        if not txt:                       # never ship an empty summary
            txt = "The appeal is disposed of in terms of the judgment."
        sub.append({"id": rid, "summary": txt})
    assert len(sub) == len(test_rows), "one summary per test document"
    assert all(isinstance(s["summary"], str) and s["summary"] for s in sub)
    d = Path(cfg.out_dir) / "submissions" / name
    d.mkdir(parents=True, exist_ok=True)
    jp = d / "prediction.json"
    jp.write_text(json.dumps(sub, ensure_ascii=False, indent=1))
    zp = d / "submission.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(jp, "prediction.json")
    wl = [len(s["summary"].split()) for s in sub]
    print(f"wrote {jp}\n      {zp}")
    print(f"  {len(sub)} summaries | mean {np.mean(wl):.0f} words "
          f"(min {min(wl)}, max {max(wl)})")
    print(f"\n  sample: {sub[0]['summary'][:300]}")
    return sub

## 3 — Data

Measures the corpus with **your** tokenizer before committing to a sequence budget,
then tokenizes once and caches. Read the printout: if the summaries are longer than
`tgt_len`, raise it before training — a truncated target teaches the model to stop
early and costs recall on every metric.

In [14]:
cfg = P6Config(
    out_dir=str(OUT), cache_dir=str(OUT / "cache"),
    results_json=str(OUT / "results.json"),
    ckpt_p1=str(CKPT_P1), tokenizer_json=str(TOK_JSON),
    hf_token=HF_TOKEN, quick=QUICK, max_minutes=BUDGET_MIN,
)
set_seed(cfg.seed)
tok = load_tokenizer(cfg)

train_rows = load_split(cfg, "train")
dev_rows   = load_split(cfg, "dev")
print(f"\ntrain {len(train_rows):,} | dev {len(dev_rows):,}")

# ---- measure with the real tokenizer before fixing the budget ----
import random as _r
_s = _r.Random(0).sample(train_rows, min(120, len(train_rows)))
dl = np.array([len(enc_ids(tok, d)) for _, d, _ in _s])
sl = np.array([len(enc_ids(tok, s)) for _, _, s in _s if s])
print(f"\ndocument tokens : median {np.median(dl):.0f}  mean {dl.mean():.0f}  "
      f"p90 {np.percentile(dl,90):.0f}  max {dl.max():.0f}")
print(f"summary  tokens : median {np.median(sl):.0f}  mean {sl.mean():.0f}  "
      f"p90 {np.percentile(sl,90):.0f}  max {sl.max():.0f}")
print(f"compression     : summary / document = {sl.mean()/max(dl.mean(),1):.3f}")
print(f"\nwith src_len={cfg.src_len}: the encoder sees all of "
      f"{(dl <= cfg.src_len).mean():.0%} of documents, "
      f"and {min(1.0, cfg.src_len/max(np.median(dl),1)):.0%} of the median one")
_cov = (sl <= cfg.tgt_len - 1).mean()
print(f"with tgt_len={cfg.tgt_len}: {_cov:.0%} of reference summaries fit whole")
if _cov < 0.9:
    print(f"  ^ RAISE cfg.tgt_len (try {int(np.percentile(sl,95))+8}) — a truncated "
          f"target trains the model to stop early")

data = prepare_data(cfg, tok, ("train", "dev"))
Str, TItr, TOtr, sl_tr, tl_tr, ids_tr, sum_tr = data["train"]
Sdv, TIdv, TOdv, sl_dv, tl_dv, ids_dv, sum_dv = data["dev"]
TRAIN_LOADER = make_loader(Str, TItr, TOtr, cfg.batch_size, True, cfg.seed)
DEV_LOADER   = make_loader(Sdv, TIdv, TOdv, cfg.batch_size, False, cfg.seed)

# a fixed dev slice for end-of-run ROUGE; generation is the expensive part
N_DEV_GEN = 40 if QUICK else 150
GEN_IDX = np.arange(min(N_DEV_GEN, len(Sdv)))
print(f"\nROUGE will be computed on {len(GEN_IDX)} dev documents per experiment")
print(f"effective batch {cfg.batch_size * cfg.grad_accum} | "
      f"{math.ceil(len(Str)/cfg.batch_size/cfg.grad_accum):,} steps/epoch | "
      f"cap {cfg.max_minutes} min per run")

tokenizer: vocab 40,000  specials None
  PAD=0 UNK=1 BOS=2 EOS=3 (</s> was never used in pretraining — this phase trains it)


Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

  [train] Phase_6_Summ/Train/train.json
  no dev.json in the dataset — held out 351 of 7030 training pairs as dev (seed 42). train is now 6679 pairs, disjoint from dev.

train 6,679 | dev 351

document tokens : median 3536  mean 4596  p90 8281  max 34808
summary  tokens : median 671  mean 860  p90 1423  max 6827
compression     : summary / document = 0.187

with src_len=1024: the encoder sees all of 3% of documents, and 29% of the median one
with tgt_len=1024: 78% of reference summaries fit whole
  ^ RAISE cfg.tgt_len (try 1647) — a truncated target trains the model to stop early
[cache hit ] train: 6679 pairs (p6_train_head_1024_1024.npz)
[cache hit ] dev: 351 pairs (p6_dev_head_1024_1024.npz)

ROUGE will be computed on 150 dev documents per experiment
effective batch 16 | 418 steps/epoch | cap 90 min per run


In [15]:
def rebuild(bc, ck):
    """Reconstruct any model this notebook saved, from its checkpoint alone."""
    m = EncDecGQA(bc)
    p = ck.get("peft") or {}
    if p.get("kind") == "lora":
        inject_lowrank(attn_sites(m, p["sites"]), LoRALinear, p["r"], p["alpha"])
    elif p.get("kind") == "adapter":
        inject_adapters(m, p["bottleneck"], p["sites"],
                        p.get("variant", "houlsby"))
    miss, unexp = m.load_state_dict(ck["model"], strict=False)
    miss = [x for x in miss if "rope_" not in x]
    if miss or unexp:
        raise RuntimeError(f"rebuild mismatch: missing {miss[:4]} "
                           f"unexpected {unexp[:4]}")
    return m


def finish(name, technique, model, bc, peft, info, tr, tot, extra=None,
           gen_batch=8):
    """Score on the dev slice, save, record. Generation is the only slow part."""
    cands = generate(model, Sdv[GEN_IDX], cfg, batch_size=gen_batch,
                     desc=f"{name} dev")
    texts = [dec_ids(tok, c) for c in cands]
    m = score_summaries([sum_dv[i] for i in GEN_IDX], texts)
    print(f"\n[{name}] dev loss {info['best_val_loss']:.4f}  "
          f"ROUGE-1 {m.get('rouge1',0):.4f}  ROUGE-L {m.get('rougeL',0):.4f}  "
          f"len {m.get('gen_len',0):.0f}w (ref {m.get('ref_len',0):.0f}w)  "
          f"repetition {m.get('repetition',0):.3f}")
    print(f"  sample: {texts[0][:260]}")
    lat = None
    try:
        t0 = time.time(); generate(model, Sdv[:4], cfg, batch_size=4)
        lat = (time.time() - t0) / 4
    except Exception:
        pass
    costs = {"sec_per_summary": round(lat, 3) if lat else None,
             "model_mb": round(sum(v.numel()*v.element_size()
                                   for v in model.state_dict().values())/1e6, 1)}
    save_run(cfg, name, model, bc, technique, peft, info, m, tr, tot, extra)
    (Path(cfg.out_dir)/"analysis").mkdir(parents=True, exist_ok=True)
    (Path(cfg.out_dir)/"analysis"/f"{name}.json").write_text(
        json.dumps({"name": name, "technique": technique, "metrics": m,
                    "samples": texts[:10],
                    "refs": [sum_dv[i] for i in GEN_IDX][:10]},
                   indent=1, default=float))
    row = record(cfg, name, technique, info, m, tr, tot, costs, extra)

    # everything this experiment produced, in one folder named after it:
    # the metrics, ten generated/reference pairs, and the notebook that made them
    exp = Path(cfg.out_dir) / "experiments" / name
    exp.mkdir(parents=True, exist_ok=True)
    (exp / "result.json").write_text(json.dumps(row, indent=1, default=float))
    (exp / "samples.json").write_text(json.dumps(
        [{"generated": t, "reference": sum_dv[i]}
         for t, i in zip(texts[:10], GEN_IDX[:10])], indent=1, ensure_ascii=False))
    save_notebook(exp, stem=name)
    print(f"  experiment folder -> {exp}")
    return row


print("runner ready — every experiment below is idempotent and time-capped")

runner ready — every experiment below is idempotent and time-capped


### E4 / E5 — bottleneck adapters: Houlsby vs Pfeiffer placement

The task statement lists these as two techniques, and they are: Houlsby puts an
adapter after **every** sublayer, so it can correct the attention output before the
FFN sees it; Pfeiffer puts **one** after the FFN, so it can only correct what comes
out of the whole block. Same bottleneck, roughly half the parameters for Pfeiffer.

Both zero-initialise the up-projection, so the adapted model **is** the frozen model
at step 0 — asserted on real logits before training starts.

In [ ]:
for NAME, VARIANT in (("E4_houlsby", "houlsby"), ("E5_pfeiffer", "pfeiffer")):
    set_seed(cfg.seed)
    ec = replace(cfg, lr=3e-4)
    peft = {"kind": "adapter", "bottleneck": 64, "sites": ["enc", "dec"],
            "variant": VARIANT}
    model, bc = load_phase1(ec, verbose=False)
    model.eval()
    with torch.no_grad():
        s_ = torch.from_numpy(Sdv[:2].astype(np.int64))
        ti_ = torch.from_numpy(TIdv[:2].astype(np.int64))
        before = model(s_, ti_, src_kpm=(s_ == PAD_ID)).clone()
    ads = inject_adapters(model, peft["bottleneck"], peft["sites"], VARIANT)
    model.eval()
    with torch.no_grad():
        after = model(s_, ti_, src_kpm=(s_ == PAD_ID))
    print(f"  [{NAME}] {len(ads)} adapters | identity at init: "
          f"max|delta logit| = {(after-before).abs().max():.2e}")
    freeze_all(model)
    for a in ads:
        for p in a.parameters():
            p.requires_grad = True
    tr, tot = param_report(model, f"[{NAME}] ")
    start, skip = resume_into(model, ec, NAME, TRAIN_LOADER)
    if not skip:
        info = train(model, TRAIN_LOADER, DEV_LOADER, ec, label=NAME,
                     start_step=start)
        finish(NAME, f"{VARIANT.capitalize()} adapters b=64", model, bc, peft,
               info, tr, tot)
    del model; torch.cuda.empty_cache()

  [E4_houlsby] 25 adapters | identity at init: max|delta logit| = 0.00e+00
[E4_houlsby] trainable 2,081,600 / 79,308,480 = 2.625%
  [E4_houlsby] step 250/5016 val_loss 3.1029 ppl 22.3  *best*  (2.7 min)
  [E4_houlsby] step 500/5016 val_loss 3.0626 ppl 21.4  *best*  (5.4 min)
  [E4_houlsby] step 750/5016 val_loss 3.0373 ppl 20.8  *best*  (8.1 min)
  [E4_houlsby] step 1000/5016 val_loss 3.0202 ppl 20.5  *best*  (10.8 min)
  [E4_houlsby] step 1250/5016 val_loss 3.0099 ppl 20.3  *best*  (13.4 min)
  [E4_houlsby] step 1500/5016 val_loss 3.0059 ppl 20.2  *best*  (16.1 min)
  [E4_houlsby] step 1750/5016 val_loss 3.0039 ppl 20.2  *best*  (18.8 min)
  [E4_houlsby] step 2000/5016 val_loss 2.9998 ppl 20.1  *best*  (21.5 min)
  [E4_houlsby] step 2250/5016 val_loss 2.9919 ppl 19.9  *best*  (24.2 min)
  [E4_houlsby] step 2500/5016 val_loss 2.9919 ppl 19.9  (26.9 min)
  [E4_houlsby] step 2750/5016 val_loss 2.9896 ppl 19.9  *best*  (29.6 min)
  [E4_houlsby] step 3000/5016 val_loss 2.9969 ppl 20.0  (32

---
Next: `p6_05_distillation`.